# IST Scenario 2 — Did early heparin help, and at what cost in bleeding?

_Weighing benefits against harms in a trial with a three-level exposure. Level: second analysis. About 90 minutes._

**Learning goals.** Compare several outcomes across a treatment with more than two levels;
recognise a dose-response relationship; put a benefit and a harm on the same scale (events per
1,000 patients) and reach a net judgement; explain what a factorial trial is.

**Clinical background.** In 1991 many physicians gave heparin routinely after an ischaemic
stroke, hoping to stop the thrombus growing and to prevent recurrence and pulmonary embolism.
Others feared haemorrhagic transformation. IST tested two doses of subcutaneous heparin,
5,000 IU twice daily (low) and 12,500 IU twice daily (medium), against no heparin, for 14 days.
Because aspirin was tested at the same time, every patient sits in one of six cells: a 3 x 2
factorial design, in which each drug can be analysed as if the other did not exist.

**Today.** Routine early anticoagulation after ischaemic stroke is not recommended, largely
because of IST. Read this scenario as the origin of that practice, not as a guide to it.

**Before you start.** IST randomised 19,435 patients between 1991 and 1996. Nobody received
thrombolysis or thrombectomy, one third were randomised before a CT scan, and "dependent" was
measured by questionnaire, not by the modified Rankin Scale. The *conclusions* of the trial still
hold and are current practice; the *absolute percentages* describe 1990s stroke care and must not
be quoted to patients.

**How this notebook works.** Every analysis is a *federated* call: the code selects variables,
defines a cohort with filters and asks the MIP for aggregates (counts, means, a test, a model).
No patient row ever reaches this notebook. Run the cells in order; each step says what to look
for and what result to expect.

## 1. Connect to the MIP and open the IST pathology

In [ ]:
import mip
from mip.filters import F

client = mip.Client.from_env()          # connection configured by the MIP portal
catalog = client.catalog()

try:
    dm = catalog.data_model("IST")      # add version="1.0" if the catalogue holds several versions
except LookupError:
    display(catalog.summaries())        # the pathology label differs: pick it from this list
    raise
ist = dm.datasets.list()[0]             # the IST pathology holds a single dataset
display(dm)
display(ist)

## 2. Toolbox

In [ ]:
# ---- Toolbox: small helpers that turn MIP results into readable tables and charts. ----
# Run this cell once. You do not need to read it to follow the scenario; come back to it
# when you want to see how a result payload is unpacked.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from mip.preprocessing import MissingValuesHandler


def labels_of(variable):
    """{code: label} of a categorical variable, whatever format the catalogue uses."""
    raw = variable._data.get("enumerations") or {}
    if isinstance(raw, dict):
        return {str(k): str(v) for k, v in raw.items()}
    out = {}
    for item in raw:
        if isinstance(item, dict):
            out[str(item.get("code"))] = str(item.get("label"))
        else:
            out[str(item)] = str(item)
    return out


def analysis(variables, filters=None, drop_missing=False):
    """A Pipeline on the IST dataset with the given variables and cohort filter.
    Tests and models need drop_missing=True: the platform requires an explicit missing-values
    step and analyses complete cases only. describe() keeps it off so it can count missing values."""
    handler = MissingValuesHandler(strategies={v: "drop" for v in variables}) if drop_missing else None
    return mip.Pipeline(analysis_set=dm.select(datasets=[ist], variables=variables), filters=filters,
                        handle_missing=handler)


def rows_for(result, variable):
    """The pooled 'describe' row of one variable (the platform adds one row per dataset as well)."""
    return [r for r in result.raw.get("featurewise", [])
            if r.get("variable") == variable._code and r.get("dataset") == "all datasets"]


def numeric_summary(result, variables):
    """Mean, SD, median and range of numeric variables from a describe result."""
    rows = []
    for v in variables:
        for r in rows_for(result, v):
            d = r.get("data") or {}
            rows.append({"variable": v.label, "n": d.get("num_dtps"), "missing": d.get("num_na"),
                         "mean": d.get("mean"), "sd": d.get("std"), "median": d.get("q2"),
                         "min": d.get("min"), "max": d.get("max")})
    return pd.DataFrame(rows)


def category_counts(result, variable):
    """Counts and percentages of the levels of a categorical variable from a describe result."""
    lab = labels_of(variable)
    rows = []
    for r in rows_for(result, variable):
        counts = (r.get("data") or {}).get("counts") or {}
        total = sum(counts.values())
        for c, n in counts.items():
            rows.append({"variable": variable.label, "level": lab.get(str(c), str(c)), "count": n,
                         "percent": 100 * n / total if total else np.nan})
    return pd.DataFrame(rows)


def describe_by(variables, group, base_filter=None):
    """Describe the variables separately in each level of `group` (one federated call per level).
    Returns (numeric table, categorical table) with a column per group level."""
    num, cat = [], []
    for c, label in labels_of(group).items():
        f = F(group) == c
        if base_filter is not None:
            f = f & base_filter
        res = analysis(variables, f).describe()
        n = numeric_summary(res, [v for v in variables if v.is_numerical()])
        if len(n):
            num.append(n.assign(group=label))
        for v in variables:
            if v.is_categorical():
                k = category_counts(res, v)
                if len(k):
                    cat.append(k.assign(group=label))
    num = pd.concat(num, ignore_index=True) if num else pd.DataFrame()
    cat = pd.concat(cat, ignore_index=True) if cat else pd.DataFrame()
    if len(num):
        num = num.pivot(index="variable", columns="group", values=["n", "mean", "sd"])
    if len(cat):
        cat = cat.pivot(index=["variable", "level"], columns="group", values="percent")
    return num, cat


def crosstab(exposure, outcome, base_filter=None, event="1"):
    """Cross-tabulation of outcome by exposure, as the MIP does it: counts come from one
    describe per exposure level, the p-value from the Chi-squared Test on the whole cohort."""
    lab_x, lab_y = labels_of(exposure), labels_of(outcome)
    rows = []
    for c, label in lab_x.items():
        f = (F(exposure) == c) & F(outcome).is_not_null()
        if base_filter is not None:
            f = f & base_filter
        res = analysis([exposure, outcome], f).describe()
        counts = {}
        for r in rows_for(res, outcome):
            counts = (r.get("data") or {}).get("counts") or {}
        n = sum(counts.values())
        if n == 0:
            continue
        row = {exposure.label: label, "n": n}
        for yc, ylabel in lab_y.items():
            row[ylabel] = counts.get(yc, 0)
        row["% " + lab_y.get(event, event)] = 100 * counts.get(event, 0) / n
        rows.append(row)
    table = pd.DataFrame(rows)
    f = F(exposure).is_not_null() & F(outcome).is_not_null()
    if base_filter is not None:
        f = f & base_filter
    chi = analysis([exposure, outcome], f, drop_missing=True).chi_square_test(x=exposure, y=outcome)
    stats = chi.raw if isinstance(chi.raw, dict) else {}
    table.attrs["chi2"] = stats.get("chi2")
    table.attrs["p"] = stats.get("p_value", stats.get("p"))
    table.attrs["outcome"] = outcome.label
    return table


def fmt_p(p):
    if p is None or (isinstance(p, float) and np.isnan(p)):
        return "n/a"
    return "< 0.001" if p < 0.001 else f"{p:.3f}"


def show_crosstab(table, title=None):
    """Display a crosstab table, its chi-square test and a bar chart of the percentages."""
    pct_col = [c for c in table.columns if c.startswith("% ")][0]
    display(table.style.format({pct_col: "{:.1f}"}).hide(axis="index"))
    print(f"Chi-squared Test: chi2 = {table.attrs['chi2']:.2f}, p = {fmt_p(table.attrs['p'])}")
    ax = table.set_index(table.columns[0])[pct_col].plot(kind="bar", color="#4C72B0", figsize=(5, 3))
    ax.set_ylabel(pct_col)
    ax.set_title(title or f"{table.attrs['outcome']} by {table.columns[0]}")
    ax.set_xlabel("")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()


def odds_ratio_table(result, variables):
    """Logistic regression result -> one row per term with odds ratio, 95% CI and p.
    Categorical covariates appear as 'variable = level' relative to the reference level
    (the first level of the variable) that the platform leaves out."""
    by_code = {v._code: v for v in variables}

    def pretty(term):
        term = str(term)
        if "[" in term and term.endswith("]"):
            base, level = term[:-1].split("[", 1)
            v = by_code.get(base)
            return f"{v.label} = {labels_of(v).get(level, level)}" if v else term
        v = by_code.get(term)
        return f"{v.label} (per unit)" if v else term

    frame = result.to_frame().copy()
    frame = frame[~frame["feature"].astype(str).str.lower().isin(["intercept", "const"])]
    frame["term"] = frame["feature"].map(pretty)
    for col in ("odds_ratio", "or_lower_ci", "or_upper_ci", "p"):
        if col not in frame.columns:
            frame[col] = np.nan
    out = frame[["term", "odds_ratio", "or_lower_ci", "or_upper_ci", "p"]]
    return out.rename(columns={"or_lower_ci": "ci_low", "or_upper_ci": "ci_high"}).reset_index(drop=True)


def show_odds_ratios(table, title):
    """Table and forest plot (log scale) of odds ratios."""
    display(table.style.format({"odds_ratio": "{:.3f}", "ci_low": "{:.3f}", "ci_high": "{:.3f}", "p": "{:.4f}"}).hide(axis="index"))
    t = table.iloc[::-1]
    fig, ax = plt.subplots(figsize=(8, 0.5 * len(t) + 1.5))
    ax.errorbar(t["odds_ratio"], range(len(t)),
                xerr=[t["odds_ratio"] - t["ci_low"], t["ci_high"] - t["odds_ratio"]], fmt="o", capsize=4)
    ax.axvline(1, linestyle="--", linewidth=1)
    ax.set_xscale("log")
    ax.set_yticks(range(len(t)))
    ax.set_yticklabels(t["term"])
    ax.set_xlabel("odds ratio (log scale)")
    ax.set_title(title)
    plt.tight_layout()
    plt.show()


print("Toolbox ready.")

## 3. The variables of this scenario

| Role | Variable in the tree | Codes |
|---|---|---|
| exposure, 3 levels | Heparin allocated (`RXHEP`) | 0 avoid, 1 low dose, 2 medium dose |
| exposure, 2 levels | Any heparin allocated (`RXHEPANY`, curation-derived) | 0, 1 |
| the six factorial cells | Treatment group (`TRTGRP`, curation-derived) | 1 to 6 |
| the other factor | Aspirin allocated (`RXASP`) | 0, 1 |
| primary outcome | Dead or dependent at 6 months (`DEADDEP6`) | 0, 1 |
| early outcome | Died within 14 days (`ID14`) | 0, 1 |
| expected benefits | Ischaemic stroke within 14 days (`ISC14`), Pulmonary embolism within 14 days (`PE14`) | 0, 1 |
| expected harms | Cerebral bleed within 14 days (`H14`), Major non-cerebral bleed within 14 days (`TRAN14`) | 0, 1 |

In [ ]:
rxhep    = dm.variables["Heparin allocated"]
rxhepany = dm.variables["Any heparin allocated"]
trtgrp   = dm.variables["Treatment group"]
rxasp    = dm.variables["Aspirin allocated"]
dd6      = dm.variables["Dead or dependent at 6 months"]
id14     = dm.variables["Died within 14 days"]
isc14    = dm.variables["Ischaemic stroke within 14 days"]
pe14     = dm.variables["Pulmonary embolism within 14 days"]
h14      = dm.variables["Cerebral bleed within 14 days"]
tran14   = dm.variables["Major non-cerebral bleed within 14 days"]

display(mip.to_frame([rxhep, rxhepany, trtgrp, rxasp, dd6, id14, isc14, pe14, h14, tran14]))

## Step 1. See the factorial design

*Expected:* the four cells that include heparin hold about 2,430 patients each, the two cells
without heparin about 4,860 each. Crossing aspirin against heparin shows that every heparin
group has exactly half of its patients on aspirin: this balance is what allows each drug to be
analysed separately.

In [ ]:
design = analysis([trtgrp]).describe()
cells = category_counts(design, trtgrp)
display(cells.style.format({"percent": "{:.1f}"}).hide(axis="index"))
ax = cells.set_index("level")["count"].plot(kind="barh", figsize=(7, 3), color="#4C72B0")
ax.set_xlabel("patients")
ax.set_ylabel("")
ax.set_title("The six cells of the 3 x 2 factorial design")
plt.tight_layout()
plt.show()

# aspirin within each heparin group: the percentages should all be 50%
show_crosstab(crosstab(rxhep, rxasp), "Aspirin allocation within each heparin group")

## Step 2. Primary outcome, heparin versus no heparin

*Expected:* 6,063 of 9,641 (62.9%) dead or dependent with heparin versus 6,062 of 9,644 (62.9%)
without. Identical, p = 0.97. This is the Lancet's result to the decimal.

In [ ]:
show_crosstab(crosstab(rxhepany, dd6), "Dead or dependent at 6 months, any heparin vs none")

## Steps 3 and 4. Primary outcome and early death, by dose

*Expected:* dead or dependent 62.9% (avoid), 63.1% (low), 62.6% (medium), p = 0.88. Died within
14 days 9.3%, 8.7%, 9.3%, p = 0.47. No dose does better on either outcome.

In [ ]:
show_crosstab(crosstab(rxhep, dd6), "Dead or dependent at 6 months, by heparin dose")
show_crosstab(crosstab(rxhep, id14), "Died within 14 days, by heparin dose")

## Steps 5 and 6. The expected benefits and the harms, by dose

*Expected:*

| Event within 14 days | Avoid (n 9,718) | Low dose (n 4,861) | Medium dose (n 4,856) | p |
|---|---|---|---|---|
| Recurrent ischaemic stroke | 223 (2.3%) | 71 (1.5%) | 101 (2.1%) | 0.003 |
| Pulmonary embolism | 81 (0.8%) | 33 (0.7%) | 20 (0.4%) | 0.015 |
| Intracranial haemorrhage | 41 (0.4%) | 35 (0.7%) | 85 (1.8%) | < 0.001 |
| Major extracranial bleed | 37 (0.4%) | 30 (0.6%) | 99 (2.0%) | < 0.001 |

Heparin did prevent some recurrent strokes and pulmonary emboli. Both harms rise with dose: a
monotonic increase is a dose-response relationship, one of the strongest arguments that the drug
causes the event.

In [ ]:
events = {"Recurrent ischaemic stroke": isc14, "Pulmonary embolism": pe14,
          "Intracranial haemorrhage": h14, "Major extracranial bleed": tran14}
tables = {name: crosstab(rxhep, v) for name, v in events.items()}

rate = pd.DataFrame({name: t.set_index("Heparin allocated")["% Yes"] * 10 for name, t in tables.items()}).T
rate.columns = [c.split(":")[0] for c in rate.columns]          # short dose labels
rate["p"] = [tables[name].attrs["p"] for name in rate.index]
rate.index.name = "events per 1,000 patients"
display(rate.style.format({c: "{:.0f}" for c in rate.columns[:-1]} | {"p": fmt_p}))

ax = rate.drop(columns="p").plot(kind="bar", figsize=(8, 4), color=["#8C8C8C", "#4C72B0", "#C44E52"])
ax.set_ylabel("events per 1,000 patients")
ax.set_title("14-day events by heparin dose: benefits (left) and harms (right)")
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.show()

## Step 7. Put benefit and harm on one scale

For the medium dose versus no heparin, per 1,000 patients: events prevented (ischaemic stroke,
pulmonary embolism) against events caused (intracranial haemorrhage, major bleed). The code
below computes the balance sheet from the tables of Step 5 and 6.

*Expected:* roughly 2 ischaemic strokes and 4 pulmonary emboli prevented against 13 intracranial
haemorrhages and 16 major bleeds caused. The harms outnumber the benefits, and an intracranial
haemorrhage is at least as disabling as an ischaemic stroke. That is why the net effect on death
or dependency is zero.

In [ ]:
avoid, low, medium = rate.columns[:3]
balance = pd.DataFrame({
    "medium dose vs avoid, per 1,000": rate[medium] - rate[avoid],
    "low dose vs avoid, per 1,000": rate[low] - rate[avoid],
})
balance["reading"] = np.where(balance.iloc[:, 0] < 0, "prevented", "caused")
display(balance.style.format({balance.columns[0]: "{:+.1f}", balance.columns[1]: "{:+.1f}"}))

ax = balance.iloc[:, 0].plot(kind="barh", figsize=(7, 3), color=np.where(balance.iloc[:, 0] < 0, "#55A868", "#C44E52"))
ax.axvline(0, color="black", linewidth=1)
ax.set_xlabel("extra events per 1,000 patients (negative = prevented)")
ax.set_ylabel("")
ax.set_title("Balance sheet of medium-dose heparin")
plt.tight_layout()
plt.show()

## Step 8. State the result

> In IST, subcutaneous heparin started within 48 hours of ischaemic stroke had no effect on death
> or dependency at six months (62.9% in both groups) or on 14-day mortality. It prevented a few
> early recurrent ischaemic strokes and pulmonary emboli, but caused a dose-dependent excess of
> intracranial haemorrhage (0.4%, 0.7%, 1.8% for no, low and medium dose) and of major
> extracranial bleeding that cancelled the benefit. This is the origin of the current
> recommendation against routine early anticoagulation in acute ischaemic stroke.

**Questions for discussion**

1. Steps 2 to 4 show "no effect". Is absence of a significant difference the same as evidence of
   no difference? What does the size of the trial allow you to say about how large a benefit
   could still be hidden?
2. Heparin prevented pulmonary embolism. Why is that not enough to recommend it? How would you
   prevent pulmonary embolism in these patients today?
3. Aspirin (Scenario 1) also increased extracranial bleeding, yet its net effect was positive.
   Compare the two balance sheets.

## Optional extension: does the harm of heparin depend on aspirin?

Cross-tabulate the six treatment cells against intracranial haemorrhage and compare medium-dose
heparin with and without aspirin. Then model the bleed on both factors with *Logistic
Regression*: this is how trialists check whether the two factors of a factorial trial interact
(here, only additively; the platform does not fit interaction terms).

In [ ]:
show_crosstab(crosstab(trtgrp, h14), "Cerebral bleed within 14 days, by treatment cell")

covariates = [rxhep, rxasp]
bleed_model = analysis([h14] + covariates, F(h14).is_not_null(), drop_missing=True).logistic_regression(
    x=covariates, y=h14, positive_class="1")
show_odds_ratios(odds_ratio_table(bleed_model, covariates), "Odds of cerebral bleed within 14 days")